# アニメ一枚絵 → パーツPSD
BF16対応GPUとPython 3.12のColab専用環境を使います。元画像はこのノートブックへ選択アップロードします。Drive全体のマウント・GitHubトークンは不要です。
新しい有料契約を作らず、認証済みの許可されたアカウントで実行してください。モデルカードと素材の権利を実行前に確認します。


In [ ]:
# See-through公式READMEどおりの専用環境をColab内に作る
import sys, subprocess, json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path('/content/avatar-lab')
ROOT.mkdir(exist_ok=True)
RUN = ROOT / 'output' / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN.mkdir(parents=True, exist_ok=False)
LOG = RUN / 'setup.log'
SEE = ROOT / 'see-through'
VENV = ROOT / '.venv-seethrough'
PY = VENV / 'bin' / 'python'
SEE_COMMIT = 'a25a5498e031dc7fe01232d05dadaf67736277fb'

def run(args: list[str], cwd: Path | None = None) -> None:
    print('実行:', ' '.join(args), flush=True)
    result = subprocess.run(args, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    with LOG.open('a', encoding='utf-8') as handle:
        handle.write('\n$ ' + ' '.join(args) + '\n' + result.stdout)
    print(result.stdout[-4000:], flush=True)
    result.check_returncode()

assert sys.version_info[:2] == (3, 12)
run(['nvidia-smi'])
if not SEE.exists():
    run(['git', 'clone', '--no-checkout', 'https://github.com/shitagaki-lab/see-through.git', str(SEE)])
    run(['git', 'checkout', SEE_COMMIT], SEE)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=SEE, text=True).strip() == SEE_COMMIT
for name in ('README.md', 'LICENSE'):
    (RUN / ('see-through-' + name)).write_text((SEE / name).read_text(), encoding='utf-8')
if not PY.exists() or subprocess.run([str(PY), '-m', 'pip', '--version'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT).returncode != 0:
    run([sys.executable, '-m', 'pip', 'install', 'virtualenv'])
    run([sys.executable, '-m', 'virtualenv', str(VENV)])
run([str(PY), '-m', 'pip', 'install', 'torch==2.8.0+cu128', 'torchvision==0.23.0+cu128', 'torchaudio==2.8.0+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run([str(PY), '-m', 'pip', 'install', '-r', 'requirements.txt'], SEE)
if not (SEE / 'assets').exists():
    (SEE / 'assets').symlink_to(SEE / 'common' / 'assets', target_is_directory=True)
run([str(PY), '-c', "import torch; print(torch.__version__); print(torch.cuda.get_device_name()); assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(), 'BF16対応GPUが必要です'"])
(RUN / 'pip-freeze.txt').write_text(subprocess.check_output([str(PY), '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
print('セットアップ完了:', RUN)


In [ ]:
from google.colab import files
from PIL import Image
import hashlib, io
uploaded = files.upload()
assert len(uploaded)==1, '入力PNGを1枚だけ選んでください'
image_bytes=next(iter(uploaded.values()))
with Image.open(io.BytesIO(image_bytes)) as image:
    assert image.format=='PNG' and image.mode=='RGBA', 'RGBA PNGを用意してください'
    alpha=image.getchannel('A')
    assert alpha.getextrema()[0]==0 and alpha.getbbox(), '透明背景で空でない画像が必要です'
EXPECTED_SHA256=hashlib.sha256(image_bytes).hexdigest()
INPUT=RUN/'input';INPUT.mkdir()
SRC=INPUT/'source.png';SRC.write_bytes(image_bytes)
print('入力SHA256:',EXPECTED_SHA256)


In [ ]:
# Colabでも、ローカルGPUと同じ生成入口を使う。
GENERATOR=RUN/'generate-psd.py'
GENERATOR.write_text('"""実行先に依存せず、準備済みSee-through環境から新規出力へPSDを生成する。"""\nimport argparse\nimport hashlib\nimport json\nimport subprocess\nimport time\nfrom pathlib import Path\n\nCOMMIT = "a25a5498e031dc7fe01232d05dadaf67736277fb"\n\n\ndef generate(python: Path, checkout: Path, image: Path, output: Path, timeout: int = 1800) -> None:\n    python, checkout, image, output = (path.resolve() for path in (python, checkout, image, output))\n    if output.exists():\n        raise FileExistsError(f"既存出力は上書きしません: {output}")\n    if not 1 <= timeout <= 3600:\n        raise ValueError("実行枠は1〜3600秒で指定してください")\n    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()\n    if actual != COMMIT:\n        raise ValueError("See-throughが固定版と異なります。変更せず停止します")\n    validator = (\n        "from PIL import Image; import sys; "\n        "im=Image.open(sys.argv[1]); "\n        "assert im.format==\'PNG\' and im.mode==\'RGBA\', \'RGBA PNGが必要です\'; "\n        "assert 128<=min(im.size)<=max(im.size)<=4096, \'画像寸法が範囲外です\'; "\n        "a=im.getchannel(\'A\'); assert a.getextrema()[0]==0 and a.getbbox(), \'透過した空でない絵が必要です\'"\n    )\n    subprocess.run([str(python), "-c", validator, str(image)], check=True)\n    output.mkdir(parents=True)\n    psd_output = output / "psd"\n    psd_output.mkdir()\n    command = [str(python), "inference/scripts/inference_psd.py", "--srcp", str(image),\n               "--save_dir", str(psd_output), "--save_to_psd", "--tblr_split", "--group_offload"]\n    metadata: dict[str, object] = {"status": "running", "source_commit": actual,\n                                   "input_sha256": hashlib.sha256(image.read_bytes()).hexdigest(),\n                                   "timeout_seconds": timeout}\n    metrics = output / "generation.json"\n    metrics.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")\n    start = time.monotonic()\n    try:\n        with (output / "inference.log").open("w", encoding="utf-8") as log:\n            process = subprocess.Popen(command, cwd=checkout, stdout=log, stderr=subprocess.STDOUT, text=True)\n            try:\n                while process.poll() is None:\n                    remaining = timeout - (time.monotonic() - start)\n                    if remaining <= 0:\n                        raise TimeoutError("実行枠を超過しました。出力とログを保持して停止します")\n                    try:\n                        process.wait(timeout=min(20, remaining))\n                    except subprocess.TimeoutExpired:\n                        print(f"PSD生成中: {round(time.monotonic() - start)}秒", flush=True)\n            finally:\n                if process.poll() is None:\n                    process.terminate()\n                    try:\n                        process.wait(timeout=10)\n                    except subprocess.TimeoutExpired:\n                        process.kill()\n                        process.wait(timeout=10)\n            if process.returncode != 0:\n                raise subprocess.CalledProcessError(process.returncode or 1, command)\n        files = sorted(psd_output.rglob("*.psd"))\n        if not files or any(path.stat().st_size == 0 for path in files):\n            raise ValueError("PSDが生成されていません。inference.logを確認してください")\n        metadata["status"] = "generated"\n        metadata["psd_files"] = [str(path.relative_to(output)) for path in files]\n        print(f"素材の生成完了: {output}。PSD画素・動き・配信は別途検査してください")\n    finally:\n        if metadata["status"] == "running":\n            metadata["status"] = "failed"\n        metadata["elapsed_seconds"] = round(time.monotonic() - start, 2)\n        metrics.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--python", type=Path, required=True)\n    parser.add_argument("--checkout", type=Path, required=True)\n    parser.add_argument("--image", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--timeout", type=int, default=1800)\n    args = parser.parse_args()\n    generate(args.python, args.checkout, args.image, args.output, args.timeout)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')
GENERATION=RUN/'generated'
subprocess.run([str(PY),str(GENERATOR),'--python',str(PY),'--checkout',str(SEE),
                '--image',str(SRC),'--output',str(GENERATION),'--timeout','1800'],check=True)
PSD_OUTPUT=GENERATION/'psd'
print((GENERATION/'generation.json').read_text(encoding='utf-8'))


In [ ]:
from IPython.display import display, Image as DisplayImage
NORMALIZER=RUN/'normalize-psd.py'
NORMALIZER.write_text('"""See-throughの左右別レイヤー名をAnime2.5DRigの番号付き規約へ変換する。"""\n\nimport argparse\nimport json\nimport re\nfrom pathlib import Path\n\nfrom psd_tools import PSDImage\n\n\ndef normalize(source: Path, destination: Path) -> None:\n    if destination.exists():\n        raise FileExistsError(f"既存PSDは上書きしません: {destination}")\n    psd = PSDImage.open(source)\n    changed: list[dict[str, str]] = []\n    for layer in psd.descendants():\n        match = re.fullmatch(r"(handwear|ears|eyebrow|eyewhite|irides|eyelash)-(l|r)", layer.name)\n        if match:\n            suffix = "1" if match.group(2) == "r" else "2"\n            new_name = f"{match.group(1)}_{suffix}"\n            changed.append({"before": layer.name, "after": new_name})\n            layer.name = new_name\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    psd.save(destination)\n    print(json.dumps({"output": str(destination), "changed": changed}, ensure_ascii=False))\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    normalize(args.input, args.output)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')
psd_files=[p for p in PSD_OUTPUT.rglob('*.psd') if not p.stem.endswith('_depth')]
assert len(psd_files)==1, f'PSD数が一意ではありません: {psd_files}'
RAW_PSD=psd_files[0]
RIG_PSD=PSD_OUTPUT/'avatar-rig.psd'
run([str(PY),str(NORMALIZER),'--input',str(RAW_PSD),'--output',str(RIG_PSD)])
run([str(PY),'-c',"import sys,json\nfrom pathlib import Path\nfrom psd_tools import PSDImage\nraw=PSDImage.open(sys.argv[1]);rig=PSDImage.open(sys.argv[2]);folder=Path(sys.argv[3])\nassert raw.size==rig.size and len(raw)==len(rig)\nassert raw.composite().tobytes()==rig.composite().tobytes(), '名前以外の合成結果が変わりました'\nrig.composite().save(folder/'psd-preview.png')\nitems=[{'name':l.name,'bbox':list(l.bbox),'visible':l.visible} for l in rig.descendants()]\n(folder/'layers.json').write_text(json.dumps({'size':list(rig.size),'layers':items},ensure_ascii=False,indent=2),encoding='utf-8')\n",str(RAW_PSD),str(RIG_PSD),str(RUN)])
display(DisplayImage(filename=str(RUN/'psd-preview.png'),width=400))
FINAL_PSD=RIG_PSD
print('初期PSD:',RIG_PSD)


In [ ]:
# 元絵に開き口がないときだけ、画風を合わせた透過の口素材を選択する。
# 素材はCodexの画像生成機能等で作成し、元PSDを描き直さない。
ADD_MOUTH=False
if ADD_MOUTH:
    sprites=files.upload()
    assert len(sprites)==1, '開き口PNGを1枚だけ選んでください'
    SPRITE=RUN/'mouth-open.png';SPRITE.write_bytes(next(iter(sprites.values())))
    ASSEMBLER=RUN/'add-speaking-mouth.py'
    ASSEMBLER.write_text('"""生成した開き口をPSDの独立レイヤーへ配置し、元レイヤーを検証する。"""\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom PIL import Image\nfrom psd_tools import PSDImage\nfrom psd_tools.api.layers import Layer, PixelLayer\n\n\ndef layer_digest(layer: Layer) -> str:\n    image = layer.topil()\n    if image is None:\n        raise ValueError(f"レイヤー画像がありません: {layer.name}")\n    return hashlib.sha256(image.tobytes()).hexdigest()\n\n\ndef assemble(source: Path, sprite_path: Path, destination: Path,\n             width: int | None = None, height: int | None = None) -> None:\n    if destination.exists():\n        raise FileExistsError(f"既存PSDは上書きしません: {destination}")\n    psd = PSDImage.open(source)\n    names = [layer.name for layer in psd]\n    if any(layer.is_group() for layer in psd) or len(names) != len(set(names)):\n        raise ValueError("平坦でレイヤー名が重複しないPSDが必要です")\n    mouths = [layer for layer in psd if layer.name == "mouth"]\n    if len(mouths) != 1:\n        raise ValueError("元のmouthレイヤーが一意に見つかりません")\n    closed = mouths[0]\n    original = {layer.name: layer_digest(layer) for layer in psd}\n    with Image.open(sprite_path) as image:\n        sprite = image.convert("RGBA")\n    alpha = sprite.getchannel("A")\n    # ほぼ透明な余白を寸法に含めず、実際の口の輪郭を基準に配置する。\n    bounds = Image.frombytes("L", alpha.size,\n                             bytes(255 if value > 16 else 0 for value in alpha.tobytes())).getbbox()\n    if bounds is None or min(alpha.tobytes()) != 0:\n        raise ValueError("透過した開き口素材が必要です")\n    # 素材の描画はimagegenで実施済み。ここではPSDへの配置だけを行う。\n    closed_image = closed.topil()\n    if closed_image is None:\n        raise ValueError("閉じ口の画素がありません")\n    closed_bounds = closed_image.getchannel("A").getbbox()\n    if closed_bounds is None:\n        raise ValueError("閉じ口が透明です")\n    target_width = width if width is not None else round((closed_bounds[2] - closed_bounds[0]) * 1.1)\n    target_height = height if height is not None else round(target_width * 0.5)\n    if not 2 <= target_width <= psd.width or not 2 <= target_height <= psd.height:\n        raise ValueError("口差分の寸法が画像の範囲外です")\n    sprite = sprite.crop(bounds).resize((target_width, target_height), Image.Resampling.LANCZOS)\n    left = round((closed.left + closed.right) / 2 - sprite.width / 2)\n    top = closed.top\n    closed.name = "mouth_close"\n    closed.visible = True\n    opened = PixelLayer.frompil(sprite, psd, name="mouth_open", top=top, left=left)\n    closed_index = list(psd).index(closed)\n    psd.insert(closed_index + 1, opened)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    psd.save(destination)\n    result = PSDImage.open(destination)\n    preserved = 0\n    for layer in result:\n        if layer.name == "mouth_open":\n            continue\n        original_name = "mouth" if layer.name == "mouth_close" else layer.name\n        if layer_digest(layer) != original[original_name]:\n            raise ValueError(f"元の画素が変化しました: {original_name}")\n        preserved += 1\n    if preserved != len(original) or len(result) != len(original) + 1:\n        raise ValueError("レイヤー数が一致しません")\n    preview = result.composite(force=True)\n    if preview is None:\n        raise ValueError("合成画像を生成できません")\n    preview.save(destination.with_suffix(".png"))\n    print(json.dumps({"output": str(destination), "preserved_layers": preserved,\n                      "layers": len(result), "mouth_bbox": list(opened.bbox),\n                      "sha256": hashlib.sha256(destination.read_bytes()).hexdigest()},\n                     ensure_ascii=False))\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input", type=Path, required=True)\n    parser.add_argument("--sprite", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--width", type=int)\n    parser.add_argument("--height", type=int)\n    args = parser.parse_args()\n    assemble(args.input, args.sprite, args.output, args.width, args.height)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')
    FINAL_PSD=PSD_OUTPUT/'avatar-speaking.psd'
    run([str(PY),str(ASSEMBLER),'--input',str(RIG_PSD),'--sprite',str(SPRITE),'--output',str(FINAL_PSD)])
print('最終PSD:',FINAL_PSD)


In [ ]:
import shutil,zipfile
AVATARS=RUN/'avatars';AVATARS.mkdir(exist_ok=True)
shutil.copyfile(FINAL_PSD,AVATARS/'avatar.psd')
ARCHIVE=RUN.parent/(RUN.name+'-avatar-assets.zip')
assert not ARCHIVE.exists(), '既存ZIPは保持してください'
with zipfile.ZipFile(ARCHIVE,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob('*')):
        if path.is_file():archive.write(path,path.relative_to(RUN))
with zipfile.ZipFile(ARCHIVE) as archive:assert archive.testzip() is None
print('ZIP SHA256:',hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())
print('このZIPにはリグ設定JSONはまだありません。ビューアで最終PSDを確認して書き出します。')
files.download(str(ARCHIVE))
